# Train and continue a MaSIF model

Choose a neural task, train on your labeled dataset, and continue an existing training run.

This notebook previews the configuration by default. Edit your paths and set `RUN_TASK = True` when you are ready to run.


## 1. Before you start

- Prepare separate training and validation datasets with labels.
- Choose `site`, `search`, `ligand`, `score`, or `alignment` below. Seed and neosurface use the alignment model; their complete workflows are not separate trainable models.
- Site uses packed labeled surfaces; search training uses the original positive/binder/negative search cache; ligand uses converted structure packets; score and alignment use their task-specific labeled feature packets.
- See [training data and examples](../USAGE.md#training-and-continuing-a-run) for preparation commands.


Use the `Python (masif_torch)` kernel. See [installation](../README.md#installation) and the [user guide](../USAGE.md).


## 2. Open the project


In [ ]:
from pathlib import Path
import importlib.util
import os
import shlex
import shutil
import subprocess
import sys
import tempfile

from omegaconf import OmegaConf

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file() and (path / "examples").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the MaSIF_torch repository or its tutorials directory.")
os.chdir(ROOT)
CLI = [sys.executable, "-m", "masif"]
RUN_TASK = False  # Change to True only after editing the paths below.
print("Repository:", ROOT)
print("Python:", sys.executable)


## 3. Choose your inputs and settings

Paths below are relative to the repository unless you provide an absolute path. Choose a separate output directory for each run.


In [ ]:
TASK = "train"
MODEL_TASK = "site"  # site, search, ligand, score, or alignment.
if MODEL_TASK not in {"site", "search", "ligand", "score", "alignment"}:
    raise ValueError("Choose one of: site, search, ligand, score, alignment.")

TRAIN_DIRECTORY = Path(f"data/{MODEL_TASK}/train")
VALIDATION_DIRECTORY = Path(f"data/{MODEL_TASK}/validation")
OUTPUT_DIRECTORY = Path(f"runs/tutorial_{MODEL_TASK}")
DEVICE = "cpu"  # Use cuda after configuring the GPU runtime.
EPOCHS = 10
RESUME_TRAINING = False
TRAINING_CHECKPOINT = OUTPUT_DIRECTORY / "last.pt"

# Hydra selects the corresponding task/model groups and dataset format.
config = OmegaConf.load(ROOT / "examples" / f"train_{MODEL_TASK}.yaml")
config.dataset.input = str(TRAIN_DIRECTORY)
config.dataset.validation = str(VALIDATION_DIRECTORY)
config.dataset.batch_size = 4 if MODEL_TASK == "search" else None
config.trainer.output = str(OUTPUT_DIRECTORY)
config.trainer.device = DEVICE
config.trainer.epochs = EPOCHS
config.trainer.resume = RESUME_TRAINING
config.trainer.checkpoint = str(TRAINING_CHECKPOINT) if RESUME_TRAINING else None
# For fine-tuning instead, set trainer.checkpoint to a compatible task checkpoint
# while keeping trainer.resume = False.


## 4. Preview your configuration

This prints the Hydra configuration without loading your input data or running the task.


In [ ]:
config_directory = tempfile.TemporaryDirectory(prefix="masif_tutorial_")
config_path = Path(config_directory.name) / "task.yaml"
OmegaConf.save(config, config_path)
command = CLI + [TASK, "--config", str(config_path)]
print(shlex.join(command))
subprocess.run(command + ["--cfg", "job"], check=True)


## 5. Run when ready

After editing the settings, set `RUN_TASK = True` in the setup cell and rerun the notebook. The checks below help identify missing inputs before starting.


In [ ]:
if RUN_TASK:
    for label, folder in (("training", TRAIN_DIRECTORY), ("validation", VALIDATION_DIRECTORY)):
        if not folder.is_dir():
            raise FileNotFoundError(f"Missing {label} directory: {folder}. Prepare labeled data before training.")
    if config.trainer.checkpoint and not Path(config.trainer.checkpoint).is_file():
        raise FileNotFoundError("Set TRAINING_CHECKPOINT to an existing compatible checkpoint.")
    subprocess.run(command, check=True)
else:
    print("Preview only. Edit your paths and set RUN_TASK = True to run.")


## 6. Next steps

To continue this run, set `RESUME_TRAINING = True`, point `TRAINING_CHECKPOINT` to its `last.pt`, and rerun. `EPOCHS` is the number of **additional** epochs on resume. Use `best.pt` for prediction: [site](site.ipynb), [search](search.ipynb), [ligand](ligand.ipynb), [score](score.ipynb), or [alignment](alignment.ipynb).
